<a id="contents"></a>
# Pre-GTC HCLS Hackathon: Research Paper Discovery

One notebook for the workshop brief, task, sources, and experiment plan

**You will leave with:** a research question and seed collection, an inspected record, and a small experiment with a demo plan. The notebook prepares your idea; your team builds and deploys the app afterward.

### Contents

- [Hackathon brief](#brief)
- [Setup and how to use this notebook](#setup)
- [1. Understand the task](#understand-the-task)
- [2. Explore the sources](#explore-the-sources) · [PubMed](#pubmed) · [Optional Tavily](#tavily)
- [3. Shape your idea](#shape-your-idea) · [Token Factory](#token-factory) · [Check an explanation](#evidence-check)
- [Choose app deployment](#app-deployment)
- [Finish your experiment and demo plan](#next-steps)

<a id="brief"></a>
## Hackathon brief

Researchers often have a collection of papers relevant to their topic but struggle to find more through keyword searches alone. Build an app that takes this existing collection as a starting point and discovers additional relevant publications.

Choose your own approach: citation networks, embeddings, LLM analysis, web search, or a combination. For example, you could identify publications that cite multiple papers in the starting collection, then use AI to assess and explain their relevance. There is no expected solution, algorithm, or framework.

**Deliverable:** a working app deployed on [Nebius Serverless](https://docs.nebius.com/serverless/overview). Use **Nebius Token Factory** for model inference, and optionally **Tavily** for agentic search or enrichment.

Return a ranked list of new papers with useful metadata, source links, and relevance explanations. Interactive graphs, clusters, and other exploration features are welcome.

[Next: setup](#setup) · [Contents](#contents)

<a id="setup"></a>
## Setup and how to use this notebook

Open the notebook in Jupyter or your IDE to work through the Python exercises. Use **Python 3.10 or later**.

For a local Jupyter session, run this **in a terminal from the repository root** (macOS/Linux):

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install jupyterlab openai tavily-python
python -m jupyterlab workshops/pre-gtc-hcls-hackathon/research-paper-discovery.ipynb
```

Select the environment where you installed these packages as the kernel. Run cells in order with **Shift+Enter**. Most cells contain a small example you can change. Blank exercise fields have short assertions: fill them in, then rerun the cell. Write answers in Markdown cells or your team's notes.

Use one kernel for all three sections. If you restart it, rerun setup and earlier cells to restore your variables. When changing a query, rerun its search and record cells before using their results.

Organizers provide Token Factory and Tavily API keys. Paste them into the hidden prompts when running their examples.

When ready to try inference, follow the [Token Factory setup guide](../../start-here/setup.md) and [official quickstart](https://docs.tokenfactory.nebius.com/quickstart).

[Next: understand the task](#understand-the-task) · [Back: brief](#brief)

In [ ]:
import json
from getpass import getpass
from urllib.parse import urlencode
from urllib.request import urlopen

<a id="understand-the-task"></a>
## 1. Understand the task

Start with papers a researcher already finds useful. Help them discover more papers worth reading.

**You will leave with:** a clear research question, a small starting collection, and a definition of a useful result.

Use the [brief above](#brief). Start with a few papers on a topic you understand; if you are new to the field, browse [PubMed](https://pubmed.ncbi.nlm.nih.gov/). There is no required algorithm or framework.

### From a collection to a result

- **Seed papers:** the researcher's starting collection.
- **Candidates:** papers discovered from that collection.
- **Relevance:** how a candidate paper helps with the researcher's question.
- **Ranking:** the order in which you suggest reading candidates.

"New" means absent from the starting collection; a useful discovery can be an older paper. Keyword search can help, but the collection also carries clues about topics, methods, and research interests.

### What counts as relevant?

Imagine a researcher studying immune cells in melanoma using single-cell sequencing. Their seeds cover both the disease and the method.

These candidates are fictional:

| Candidate | Connection |
|---|---|
| A | Uses the same method for a different cancer. |
| B | Studies melanoma immune cells with a different method. |
| C | A broad review that cites several seed papers. |

Rank A, B, and C for a researcher seeking evidence about melanoma. Explain your first choice in one sentence. How would the order change if the goal were finding a reusable method?

There is no single correct ranking. Make the researcher's goal explicit. Edit the answers below, or keep them in your team notes.

- Ranking for evidence about melanoma:
- Why I would read the first paper:
- Ranking for a reusable method:
- My relevance rule:

In [ ]:
example_question = 'Which single-cell methods help study melanoma immune cells?'
print(example_question)

In [ ]:
research_question = ''  # Write your question, or use example_question.
assert research_question.strip(), 'Add your research question before continuing.'
print(research_question)

### Choose a small starting collection

Pick 3–5 real papers around your question. A **DOI** identifies a publication across services; a **PMID** identifies its PubMed record.

The example below covers immune-cell diversity, tumour recognition, and metastatic sites. Use it as a starting collection or replace it with your own papers.

In [ ]:
seed_links = [
    'https://doi.org/10.1126/science.aad0501',           # Immune-cell diversity
    'https://doi.org/10.1016/j.cell.2018.10.038',       # Immunotherapy response
    'https://doi.org/10.1038/s41586-021-03704-y',        # CD8 T-cell recognition
    'https://doi.org/10.1038/s41586-022-04682-5',        # CD4 T-cell function
    'https://doi.org/10.1158/1078-0432.CCR-21-1694',     # Metastatic sites
]

### Make a result useful

The brief asks for a ranked list of additional papers, metadata, source links, and relevance explanations.

For each result, help a reader answer:

- What is it? Show title, authors, year, and an identifier where available.
- Why read it? Connect it to the seeds or research question.
- What supports that explanation? Point to an abstract, passage, or verified citation link.

Exclude the seeds and repeated records of the same publication. If evidence is missing, say what you could not assess.

The sentence below is an example explanation. What connection to your question and supporting evidence would you add?

- Connection to my question:
- Evidence I would need:

In [ ]:
relevance_reason = 'Identifies T cells that recognise tumour cells.'
print(relevance_reason)

<a id="task-notes"></a>
### Your task notes

Edit this Markdown cell, or keep notes elsewhere.

- Research question:
- Seed titles and links:
- What connects the seeds:
- A useful discovery would:
- A result I would exclude:
- Evidence I want in an explanation:

**Carry forward:** `research_question` and `seed_links` remain in this kernel. Use them to choose a search query and verify paper identities in the next section.

[Next: explore the sources](#explore-the-sources) · [Back: brief](#brief)

<a id="explore-the-sources"></a>
## 2. Explore the sources

Your [research question and seeds](#task-notes) tell you what to look for. Now explore what information you can actually retrieve.

**You will leave with:** one inspected paper record and a source worth trying.

An **API** lets an app request structured data from a service. Run the small examples below in order, then change them to explore your own topic.

### Choose a source for a question

| Resource | What you can explore |
|---|---|
| [PubMed E-utilities](https://www.ncbi.nlm.nih.gov/books/NBK25501/) | Biomedical paper records and available abstracts. |
| [OpenAlex works](https://help.openalex.org/data/works/attributes/) | Paper metadata, reference links, and citing papers across disciplines. |
| [Semantic Scholar API](https://api.semanticscholar.org/api-docs/) | Paper metadata, citation links, and recommendations. |
| [Tavily](https://docs.tavily.com/documentation/quickstart) | Web search for paper pages and extra context. |

PubMed and [OpenAlex](https://help.openalex.org/api/authentication/) allow basic requests without a key. [Semantic Scholar](https://webflow.semanticscholar.org/product/api) has public endpoints, though some require a key. Tavily requires one. Check each service's current limits before automating requests.

Use Tavily for web context and paper pages, and scholarly sources for citation links.

### Inspect one paper

Find one seed in [PubMed](https://pubmed.ncbi.nlm.nih.gov/), [OpenAlex](https://openalex.org/), or [Semantic Scholar](https://www.semanticscholar.org/).

Record its title, authors, year, identifier, abstract availability, and source URL. Can you confirm that it is the same publication as your seed?

Metadata means descriptive fields such as title and year. An abstract is a summary; full text is the complete paper. Availability varies. A record or PDF link does not guarantee access to full text.

<a id="pubmed"></a>
### Try the PubMed API

[ESearch](https://www.ncbi.nlm.nih.gov/books/NBK25499/#chapter4.ESearch) finds IDs. [ESummary](https://www.ncbi.nlm.nih.gov/books/NBK25499/#chapter4.ESummary) retrieves record summaries. Responses use JSON, a structured text format.

These examples use Python's standard library from [shared setup](#setup). Choose any source for your app.

A topic search returns possible **candidates**, which may differ from your seeds. Inspect their identities before using them; exclude seeds from final recommendations.

#### Search for a topic

Start with `research_question`, then adapt the query to your topic. You can also try `"melanoma[Title]"` or `"asthma[Title]"`; `[Title]` limits a search to paper titles. If there are no matches, broaden the query.

In [ ]:
query = research_question  # Or try "melanoma[Title]" or "asthma[Title]".
query

In [ ]:
endpoint = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
params = {"db": "pubmed", "term": query, "retmax": 2, "retmode": "json"}
with urlopen(endpoint + "?" + urlencode(params), timeout=20) as response:
    search = json.load(response)
search

#### Inspect the response

Look inside `esearchresult`. These examples read the returned PMIDs and compare them with the total match count. What does `retmax` limit?

In [ ]:
pmids = search["esearchresult"]["idlist"]
pmids

In [ ]:
print("Total matches:", search["esearchresult"]["count"])
print("Returned IDs:", len(pmids))

#### Choose one paper

The next cell selects the first returned PMID with `pmids[0]`. To inspect another result, replace it with an ID from `pmids`. If your search has no matches, try a broader query and rerun the search cells.

In [ ]:
assert pmids, "No papers returned. Broaden your query and rerun the search cells."
pmid = pmids[0]  # Or choose another ID from pmids.
assert pmid in pmids, "Choose an ID from pmids."

In [ ]:
endpoint = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"
params = {"db": "pubmed", "id": pmid, "retmode": "json"}
with urlopen(endpoint + "?" + urlencode(params), timeout=20) as response:
    summary = json.load(response)
list(summary)

#### Explore the record

List its fields, then inspect one. Change `field` to explore another value. The remaining cells show author names, a source link, and whether this response contains an abstract field.

In [ ]:
record = summary["result"][pmid]
list(record)

In [ ]:
field = "title"  # Example: try "pubdate" or "source".
record.get(field)

In [ ]:
[author["name"] for author in record.get("authors", [])]

In [ ]:
paper_link = f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
paper_link

In [ ]:
"abstract" in record

`False` means this ESummary response has no `abstract` field; it does not tell you whether the paper has an abstract. Open `paper_link` or use [EFetch](https://www.ncbi.nlm.nih.gov/books/NBK25499/#chapter4.EFetch) to check for an available PubMed abstract.

Which information would your idea need? How would you handle a paper whose abstract is unavailable?

### Understand the connections

For a paper A:

| Connection | Meaning |
|---|---|
| References | Papers A cites. |
| Citing papers | Papers that cite A. |
| Shared references | Two papers cite some of the same work. |

These connections can suggest candidates. A citation can support, compare, or challenge a finding; it does not establish relevance or agreement.

[OpenAlex API recipes](https://help.openalex.org/how-to/api-recipes/) show how references and citing papers differ. If a field is missing, treat it as unknown rather than proof that no connection exists.

### Explore an existing tool

[Connected Papers](https://www.connectedpapers.com) builds a visual map around a starting paper. Its connections represent similarity based on shared references and citations, rather than only direct citation links.

> **Free limit:** [Five graphs per month](https://www.connectedpapers.com/pricing). Choose seeds carefully.

Try one seed or inspect its demo. Notice what helps you choose the next paper: position, title, date, or connections.

With Token Factory, Tavily, and the paper APIs, what could you improve for your researcher? Borrow an interaction idea, or design a different one.

<a id="tavily"></a>
### Test Tavily (optional)

Use the `tavily-python` package from [setup](#setup). Paste your provided key when prompted; the input is hidden.

Run this small [web search](https://docs.tavily.com/documentation/api-reference/endpoint/search) using your research question, then try a different query. Web pages still need identity and evidence checks.

Without a Tavily key, skip the next two code cells and continue to [source notes](#source-notes).

In [ ]:
from tavily import TavilyClient

tavily = TavilyClient(api_key=getpass("Tavily API key: ").strip())

In [ ]:
tavily_query = research_question  # Example alternative: "single-cell melanoma papers".
results = tavily.search(tavily_query, max_results=2)
for result in results["results"]:
    print(result["title"], result["url"])

<a id="source-notes"></a>
### Your source notes

- Paper and source URL:
- Fields my idea needs:
- Fields this source provided:
- Missing information and how I will handle it:
- One connection worth exploring:
- Source I want to try, and why:
- One interaction idea:

**Decision checkpoint:** Can this source support your first experiment? Name one needed field you found and one missing field you will retrieve elsewhere or work without.

**Carry forward:** `record` and `paper_link` describe the selected PubMed paper. Keep its source with any abstract you use in the [evidence exercise](#evidence-check). If it is already a seed, you can use it for practice but exclude it from final recommendations.

[Next: shape your idea](#shape-your-idea) · [Back: task notes](#task-notes)

<a id="shape-your-idea"></a>
## 3. Shape your idea

You have a research question and a sense of the available data. Turn those observations into a small experiment.

**You will leave with:** an approach to test, a way to judge it, and a short demo plan.

Use your [source notes](#source-notes) and the existing `research_question`. Any approach is welcome. The required outcome is the app described in the [brief](#brief).

### Choose a signal to test

| Possible approach | Question it could help answer | Limitation to investigate |
|---|---|---|
| Citation links | Which papers connect to several seeds? | New papers may have few citations. |
| Embeddings | Which abstracts discuss similar ideas? | Similar wording may hide a different research setting. |
| LLM analysis | How does a candidate relate to the question? | An explanation can make unsupported claims. |
| Web search | What relevant paper pages are discoverable? | Results can include unrelated or duplicate pages. |

**Embeddings** represent text as numbers so you can compare similarity. A **large language model (LLM)** can interpret supplied text and generate explanations.

Choose one signal, combine several, or propose another. Decide what observation would make you change direction.

<a id="token-factory"></a>
### Test Token Factory

**Inference** means running a model on an input to get a response.

Use the `openai` package from [setup](#setup). Paste your provided key when prompted; the input is hidden.

Run a short [chat request](https://docs.tokenfactory.nebius.com/quickstart) with this [Nemotron model](https://nebius.com/services/token-factory/models/nvidia-nemotron-models-inference).

Check the response against the [PubMed documentation](https://pubmed.ncbi.nlm.nih.gov/help/#pmid): a PMID is a numeric identifier. Even a simple model reply needs verification.

Without a key, skip the next two code cells and continue to the [manual evidence check](#evidence-check). You will need Token Factory for inference in the final app.

In [ ]:
from openai import OpenAI

model = "nvidia/Nemotron-3_5-Lightning"
client = OpenAI(base_url="https://api.tokenfactory.nebius.com/v1/",
                api_key=getpass("Token Factory API key: ").strip())

In [ ]:
reply = client.chat.completions.create(
    model=model, max_tokens=1024,
    messages=[{"role": "user", "content": "What is a PMID? One sentence."}],
)
print(reply.choices[0].message.content)

<a id="evidence-check"></a>
### Check an explanation

Try the fictional example below. Which claim is unsupported? Rewrite the explanation using only the abstract, and say what remains unknown.

Then use your existing `research_question` and one **public paper abstract** in the API exercise. Start with the paper at `paper_link`, or use one of your seeds. Open the paper page to obtain an available abstract: ESummary does not supply it. If no abstract is available, choose another paper and update its source link.

Ask for one relevance reason and one uncertainty, then identify the exact text supporting each claim. This practices explanation checking; it does not create the final ranking.

You can also try the prompt in the [Token Factory Playground](https://docs.tokenfactory.nebius.com/ai-models-inference/playground). If using the Playground, run the abstract and prompt cells, copy the printed prompt into it, and skip the API request cell. Use the Playground reply for the checks below.

In [ ]:
abstract = 'A pilot with 24 adults reported greater confidence after reminders. Hospital admissions were not measured.'
explanation = 'Reminders reduced hospital admissions.'
print(abstract)
print(explanation)

In [ ]:
my_note = ''  # Rewrite the explanation using only the abstract; say what remains unknown.
assert my_note.strip(), 'Add your evidence check before continuing.'
print(my_note)

In [ ]:
abstract_source = paper_link  # Or replace with the public page for another paper.
public_abstract = """

"""  # Paste that paper's public abstract here.
assert public_abstract.strip(), 'Add a public abstract, or skip to the candidate review.'

In [ ]:
prompt = (
    "Using only the supplied abstract, give one reason the paper may be relevant "
    "to the research question and one uncertainty. Quote a short supporting passage "
    "for the relevance reason. If the abstract is insufficient, say so.\n\n"
    f"Research question: {research_question}\n\nAbstract: {public_abstract}"
)
print("Paper source:", abstract_source)
print(prompt)

In [ ]:
relevance_reply = client.chat.completions.create(
    model=model, max_tokens=1024,
    messages=[{"role": "user", "content": prompt}],
)
print(relevance_reply.choices[0].message.content)

Check the reply against the abstract before accepting it. Edit these answers, or use your team notes.

- Paper title and source (use `abstract_source`):
- Relevance claim and exact supporting passage:
- Unsupported claim or remaining uncertainty:
- How I would revise the explanation:

Use the same evidence standard when reviewing candidates below.

<a id="judge-candidates"></a>
### Judge a small set of results

Before building a large system, manually review a few candidates against your research question.

Mark each **useful**, **uncertain**, or **not useful**, and write one reason. Use this small set to compare your first idea with a simple keyword search or manual exploration.

Fill in three rows below, or use your team notes. Include a misleading match and a record with a missing abstract. Exclude any candidate already in your seed collection from the final recommendations.

| Candidate title and link | Already a seed? | Decision | Reason and supporting evidence |
|---|---|---|---|
| | | | |
| | | | |
| | | | |

- Which candidate would I read first, and why?
- Which candidate looked relevant at first but failed my criteria?

This is a quick learning check, not a required scoring system.

Start with an inspected candidate from [PubMed](#pubmed), a [Tavily](#tavily) result, or manual exploration. Keep its verified identifier and source link; check against `seed_links` before recommending it.

<a id="app-deployment"></a>
### Plan the smallest demo

Choose your input format, one manageable discovery route, and a clear result view. A list is enough to start; graphs and clusters are optional.

The brief requires Nebius Token Factory for inference and an app deployed on Nebius. Tavily is optional.

[Token Factory serverless endpoints](https://docs.tokenfactory.nebius.com/public-serverless) provide model inference. For hosting your app's interface and backend, read the [Nebius Serverless overview](https://docs.nebius.com/serverless/overview).

**Deployment checkpoint:** Compare **DevLab**, **Job**, and **Endpoint** in the overview. Which option is better for your app and demo?

<a id="next-steps"></a>
### Your idea

Finish by choosing one hypothesis and a small way to test it. Edit the answers below, or keep them in your team notes, then complete the two fields after the example.

- Source and discovery signal I will try first, and why:
- Comparison I will use (for example, keyword search):
- Evidence that would make me keep this approach:
- Evidence that would make me change direction:
- Smallest result I can show in the demo:
- App deployment choice and why:

Here is an example; your approach can be different.

In [ ]:
hypothesis = 'Showing an abstract passage may help readers judge a recommendation.'
experiment = 'Review three candidate papers with one researcher.'
comparison = 'Compare their decisions with and without the passage.'
print(hypothesis, experiment, comparison)

In [ ]:
my_hypothesis = ''  # Write the idea you want to test.
my_next_step = ''  # What small experiment would help you judge it?
assert my_hypothesis.strip() and my_next_step.strip(), 'Add your hypothesis and next experiment before finishing.'
print(my_hypothesis, my_next_step)

Build the smallest version that lets you test this idea. Let what you learn guide the next step.

Before leaving, check that you have a research question and seed collection, one inspected source record, a hypothesis and next experiment, and a plan for hosting the app on [Nebius Serverless](https://docs.nebius.com/serverless/overview) with Token Factory inference.

[Review the brief](#brief) · [Task notes](#task-notes) · [Source notes](#source-notes) · [Candidate review](#judge-candidates) · [Original README](README.md)